# Setup

In [2]:
from pathlib import Path
import os
import yaml
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "configs" / "style_dinov2.yaml").is_file():
            return candidate
    raise FileNotFoundError("Cannot find repo root.")

REPO_DIR = find_repo_root()
os.chdir(REPO_DIR)

experiment = "only_3_classes"

RUNTIME_CFG_PATH = Path(f"outputs/{experiment}/style_dinov2_runtime.yaml")
assert RUNTIME_CFG_PATH.is_file(), RUNTIME_CFG_PATH

with open(RUNTIME_CFG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

OUTPUT_ROOT = Path(cfg["outputs"]["checkpoint_path"]).parents[1]
EXPLAIN_ROOT = Path(cfg["outputs"]["explainability_root"])

print("REPO_DIR:", REPO_DIR)
print("RUNTIME_CFG_PATH:", RUNTIME_CFG_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXPLAIN_ROOT:", EXPLAIN_ROOT)

REPO_DIR: /home/kostya/projects/thesis-assyrian-relief
RUNTIME_CFG_PATH: outputs/only_3_classes/style_dinov2_runtime.yaml
OUTPUT_ROOT: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes
EXPLAIN_ROOT: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes/explainability


In [21]:
def explain_image(
    relief_id: str,
    view_index: int,
    true_class: str,
    target_classes: list[str],
    suffix: str = ".jpg",
    margin_pairs: list[str] | None = None,
    occlusion_size: int | None = None,
    stride: int | None = None,
):
    cmd = [
        "uv", "run", "python", "scripts/explain_occlusion.py",
        "--config", str(RUNTIME_CFG_PATH),
        "--relief-id", relief_id,
        "--view-index", str(view_index),
        "--suffix", suffix,
        "--true-class", true_class,
        "--target-classes", *target_classes,
    ]

    if occlusion_size is not None:
        cmd += ["--occlusion-size", str(occlusion_size)]
    if stride is not None:
        cmd += ["--stride", str(stride)]

    if margin_pairs:
        cmd += ["--margin-pairs", *margin_pairs]

    print(" ".join(f'"{x}"' if " " in x else x for x in cmd))

    import subprocess
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)

    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed with return code {result.returncode}")

In [ ]:
TARGETS = ["Ashurbanipal", "Ashurnasirpal II", "Sargon II"]

for view_idx in [1, 2, 3, 4, 7]:
    explain_image(
        relief_id="BM 124920",
        view_index=view_idx,
        true_class="Ashurbanipal",
        target_classes=TARGETS,
        margin_pairs=[
            "Ashurbanipal::Ashurnasirpal II",
            "Ashurnasirpal II::Ashurbanipal",
        ],
    )

uv run python scripts/explain_occlusion.py --config outputs/only_3_classes/style_dinov2_runtime.yaml --relief-id "BM 124920" --view-index 1 --suffix .jpg --true-class Ashurbanipal --target-classes Ashurbanipal "Ashurnasirpal II" "Sargon II" --margin-pairs "Ashurbanipal::Ashurnasirpal II" "Ashurnasirpal II::Ashurbanipal"
Using device: cuda
Image: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2/BM 124920-1.jpg
Output directory: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes/explainability/occlusion/BM_124920-1
Occlusion config: {'occlusion_size': 32, 'stride': 16, 'occlusion_value': 0.0, 'clamp_negative': False}
Image: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2/BM 124920-1.jpg
Predicted: Ashurnasirpal II (0.8983)
  Ashurbanipal: prob=0.0524, logit=-0.8508
  Ashurnasirpal II: prob=0.8983, logit=1.9918
  Sargon II: prob=0.0493, logit=-0.9105

Computing occlusion heatmap for: Ashurbanipal
Saved heatmap array: /home/kostya/projects/thesis

In [23]:
for view_idx in [1]:
    explain_image(
        relief_id="MET 32.143.3",
        view_index=view_idx,
        true_class="Ashurnasirpal II",
        target_classes=TARGETS,
        margin_pairs=[
            "Ashurbanipal::Ashurnasirpal II",
            "Ashurnasirpal II::Ashurbanipal",
        ],
    )

uv run python scripts/explain_occlusion.py --config outputs/only_3_classes/style_dinov2_runtime.yaml --relief-id "MET 32.143.3" --view-index 1 --suffix .jpg --true-class "Ashurnasirpal II" --target-classes Ashurbanipal "Ashurnasirpal II" "Sargon II" --margin-pairs "Ashurbanipal::Ashurnasirpal II" "Ashurnasirpal II::Ashurbanipal"
Using device: cuda
Image: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2/MET 32.143.3-1.jpg
Output directory: /home/kostya/projects/thesis-assyrian-relief/outputs/only_3_classes/explainability/occlusion/MET_32.143.3-1
Occlusion config: {'occlusion_size': 32, 'stride': 16, 'occlusion_value': 0.0, 'clamp_negative': False}
Image: /mnt/c/Users/chend/Desktop/My_Files/Thesis/Dataset/dataset_v2/MET 32.143.3-1.jpg
Predicted: Ashurbanipal (0.8905)
  Ashurbanipal: prob=0.8905, logit=1.8882
  Ashurnasirpal II: prob=0.0598, logit=-0.8126
  Sargon II: prob=0.0497, logit=-0.9972

Computing occlusion heatmap for: Ashurbanipal
Saved heatmap array: /home/kostya/p